# Backtest Analysis

Loads JSON result files from `backtesting/results/` and visualises:
- Equity curve
- Drawdown over time
- Trade list with P&L
- Win rate and Sharpe ratio summary

Run a backtest first to generate results:
```bash
python main.py backtest --strategy rsi_mean_revert --symbol AAPL --start 2023-01-01 --end 2024-01-01
```

In [ ]:
import json
import glob
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Path to backtest results (relative to repo root; adjust if running from elsewhere)
RESULTS_DIR = Path("../backtesting/results")

## 1. Pick a result file

In [ ]:
result_files = sorted(RESULTS_DIR.glob("*.json"))
print(f"Found {len(result_files)} result file(s):")
for i, f in enumerate(result_files):
    print(f"  [{i}] {f.name}")

In [ ]:
# Change the index to select a different file
FILE_INDEX = -1  # -1 = most recent

result_path = result_files[FILE_INDEX]
print(f"Loading: {result_path.name}")

with open(result_path) as f:
    result = json.load(f)

print("Top-level keys:", list(result.keys()))

## 2. Summary metrics

In [ ]:
summary = {
    "Strategy": result.get("strategy", "unknown"),
    "Symbol": result.get("symbol", "unknown"),
    "Period": f"{result.get('start')} to {result.get('end')}",
    "Total trades": result.get("trade_count", 0),
    "Total return ($)": f"${result.get('total_return', 0):+,.2f}",
    "Max drawdown": f"{result.get('max_drawdown', 0):.1%}",
    "Sharpe ratio": f"{result.get('sharpe_ratio', 0):.2f}",
    "Win rate": f"{result.get('win_rate', 0):.1%}",
}

pd.DataFrame.from_dict(summary, orient="index", columns=["Value"])

## 3. Equity curve

In [ ]:
# Expects result["equity_curve"] = [{"date": "YYYY-MM-DD", "equity": float}, ...]
equity_data = result.get("equity_curve", [])

if equity_data:
    eq_df = pd.DataFrame(equity_data)
    eq_df["date"] = pd.to_datetime(eq_df["date"])

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=eq_df["date"],
        y=eq_df["equity"],
        mode="lines",
        name="Portfolio equity",
        line=dict(color="royalblue", width=2),
    ))
    fig.update_layout(
        title=f"Equity curve - {result.get('strategy', '')} on {result.get('symbol', '')}",
        xaxis_title="Date",
        yaxis_title="Equity ($)",
        template="plotly_white",
    )
    fig.show()
else:
    print("No equity_curve data in this result file. The backtesting engine may not yet write it.")

## 4. Trade list

In [ ]:
# Expects result["trades"] = [{"entry_date", "exit_date", "side", "entry_price", "exit_price", "pnl"}, ...]
trades = result.get("trades", [])

if trades:
    trades_df = pd.DataFrame(trades)
    trades_df = trades_df.sort_values("entry_date")
    trades_df["pnl"] = trades_df["pnl"].map(lambda x: f"${x:+,.2f}")
    display(trades_df)
else:
    print("No trades in this result file.")

## 5. P&L distribution

In [ ]:
trades_raw = result.get("trades", [])

if trades_raw:
    pnl_series = pd.Series([t["pnl"] for t in trades_raw])

    fig = px.histogram(
        pnl_series,
        nbins=30,
        title="P&L distribution per trade",
        labels={"value": "P&L ($)", "count": "Number of trades"},
        color_discrete_sequence=["royalblue"],
        template="plotly_white",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="red")
    fig.show()

    print(f"Winning trades: {(pnl_series > 0).sum()} / {len(pnl_series)}")
    print(f"Average win:  ${pnl_series[pnl_series > 0].mean():,.2f}")
    print(f"Average loss: ${pnl_series[pnl_series <= 0].mean():,.2f}")
    print(f"Expectancy:   ${pnl_series.mean():,.2f} per trade")
else:
    print("No trades to plot.")